In [1]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [2]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [11]:
df=spark.read.format('csv')\
            .option("inferSchema", True)\
            .option('header', True)\
            .load(f"file:///home/talentum/test-jupyter/DAWPPS/json-usecases/labs/data/flim.csv")
df.printSchema()
df.show()

root
 |-- id: integer (nullable = true)
 |-- movie: string (nullable = true)

+---+--------------------+
| id|               movie|
+---+--------------------+
|  1|{ 'Title': 'Avata...|
|  2|{ 'Title': 'I Am ...|
|  3|{ 'Title': '300',...|
|  4|{ 'Title': 'The A...|
|  5|{ 'Title': 'The W...|
|  6|{ 'Title': 'Inter...|
+---+--------------------+



In [8]:
import pyspark.sql.functions as F

# Create a cleaned column first
df_fixed = df.withColumn("movie_clean", F.regexp_replace(F.col("movie"), "'", '"'))
df_fixed.show()

+---+--------------------+--------------------+
| id|               movie|         movie_clean|
+---+--------------------+--------------------+
|  1|{ 'Title': 'Avata...|{ "Title": "Avata...|
|  2|{ 'Title': 'I Am ...|{ "Title": "I Am ...|
|  3|{ 'Title': '300',...|{ "Title": "300",...|
|  4|{ 'Title': 'The A...|{ "Title": "The A...|
|  5|{ 'Title': 'The W...|{ "Title": "The W...|
|  6|{ 'Title': 'Inter...|{ "Title": "Inter...|
+---+--------------------+--------------------+



In [9]:
# Using get_json_object 
# SparkSQL function (column_name, "$.json_attribute_name")
# Extract the different json attribute and value 
# to new dataframe columns and value
df2 = df_fixed.select(
     F.col("id"),
     F.get_json_object(F.col("movie_clean"), "$.Title").alias("Title"),
     F.get_json_object(F.col("movie_clean"), "$.Rated").alias("Rated"),
     F.get_json_object(F.col("movie_clean"), "$.Year").alias("Year"),
     F.get_json_object(F.col("movie_clean"), "$.Released").alias("Released")
)
df2.show()
df2.printSchema()

+---+--------------------+-----+----+-----------+
| id|               Title|Rated|Year|   Released|
+---+--------------------+-----+----+-----------+
|  1|              Avatar|PG-13|2009|18 Dec 2009|
|  2|         I Am Legend|PG-13|2007|14 Dec 2007|
|  3|                 300|    R|2006|09 Mar 2007|
|  4|        The Avengers|PG-13|2012|04 May 2012|
|  5|The Wolf of Wall ...|    R|2013|25 Dec 2013|
|  6|        Interstellar|PG-13|2014|07 Nov 2014|
+---+--------------------+-----+----+-----------+

root
 |-- id: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Rated: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Released: string (nullable = true)



In [10]:
# Using Json Tuple function 
# json_tuple(column_name, "json_attr_1", "json_attr_2", ....)
# Extract the each attribute into new dataframe column
df3 = df_fixed.select(F.col("id"), F.json_tuple(F.col("movie_clean"),
                "Title", "Rated", "Released", "Year")
                .alias("Title", "Rated", "Released", "Year"))
df3.show()
df3.printSchema()

+---+--------------------+-----+-----------+----+
| id|               Title|Rated|   Released|Year|
+---+--------------------+-----+-----------+----+
|  1|              Avatar|PG-13|18 Dec 2009|2009|
|  2|         I Am Legend|PG-13|14 Dec 2007|2007|
|  3|                 300|    R|09 Mar 2007|2006|
|  4|        The Avengers|PG-13|04 May 2012|2012|
|  5|The Wolf of Wall ...|    R|25 Dec 2013|2013|
|  6|        Interstellar|PG-13|07 Nov 2014|2014|
+---+--------------------+-----+-----------+----+

root
 |-- id: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Rated: string (nullable = true)
 |-- Released: string (nullable = true)
 |-- Year: string (nullable = true)

